<a href="https://colab.research.google.com/github/Dharshini1701/priyadharshini/blob/main/AI_Powered_Smart_Study_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q google-generativeai



In [17]:

import google.generativeai as genai
import os, json, textwrap
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel("gemini-3.6-flash")

print("Gemini API configured successfully. Model ready: gemini-3.6-flash")

Enter your Gemini API Key: ··········
Gemini API configured successfully. Model ready: gemini-3.6-flash


In [9]:
def get_study_notes():
    print("Paste your study notes below.")
    print("Press ENTER on an empty line when you are done:\n")

    lines = []

    while True:
        line = input()

        if line.strip() == '':
            break

        lines.append(line)

    return '\n'.join(lines)


notes_text = get_study_notes()

if not notes_text.strip():
    raise ValueError("No study notes were provided. Please re-run and paste some text.")

print(f"\nCollected {len(notes_text.split())} words of study notes.")

Paste your study notes below.
Press ENTER on an empty line when you are done:

Python is a programming language. Pandas is used for data analysis. SQL is used to manage databases.


Collected 17 words of study notes.


In [18]:
def summarize_notes(text):
    prompt = f"""
    You are a study assistant. Summarize the following notes into concise,
    well-organized bullet points that highlight the key concepts, definitions,
    and any important relationships between ideas.
    Notes:
    {text}
    """
    response = model.generate_content(prompt)
    return response.text

summary = summarize_notes(notes_text)
print('=== SUMMARY ===\n')
print(summary)

=== SUMMARY ===

Here is a concise summary of your notes:

* **Python**: A general programming language.
* **Pandas**: A tool/library used for **data analysis**.
* **SQL**: A language used to **manage databases**. 

*(Contextual Relationship: These three tools form a core data tech stack—SQL retrieves data from databases, Python provides the programming environment, and Pandas analyzes the data within Python.)*


In [19]:
def generate_quiz(text, num_questions=5):
    prompt = f"""
    Create {num_questions} multiple choice questions to test understanding
    of the notes below. Return ONLY valid JSON — a list of objects, each with
    keys: "question", "options" (a dict with keys A, B, C, D), and "answer"
    (the correct option letter).
    Notes:
    {text}
    """
    response = model.generate_content(prompt)
    raw = response.text.strip().strip('`').replace('json', '', 1).strip()
    try:
        quiz = json.loads(raw)
    except json.JSONDecodeError:
        quiz = raw
    return quiz

quiz = generate_quiz(notes_text, num_questions=5)

print('=== QUIZ ===\n')

if isinstance(quiz, list):
    for i, q in enumerate(quiz, 1):
        print(f"Q{i}. {q['question']}")
        for k, v in q['options'].items():
            print(f" {k}) {v}")
        print(f" Correct Answer: {q['answer']}\n")
else:
    print(quiz)

=== QUIZ ===

Q1. What is Python according to the provided text?
 A) A database system
 B) A programming language
 C) A tool used for data analysis
 D) A database management language
 Correct Answer: B

Q2. What is Pandas primarily used for?
 A) Managing databases
 B) Web development
 C) Data analysis
 D) Operating system design
 Correct Answer: C

Q3. Which technology is used to manage databases?
 A) Pandas
 B) Python
 C) SQL
 D) HTML
 Correct Answer: C

Q4. Based on the notes, which statement is correct?
 A) Python is used to manage databases.
 B) Pandas is used to manage databases.
 C) SQL is used for data analysis.
 D) Pandas is used for data analysis.
 Correct Answer: D

Q5. What is the function of SQL as described in the notes?
 A) It is a programming language.
 B) It is used to manage databases.
 C) It is used for data analysis.
 D) It is used to build web applications.
 Correct Answer: B



In [ ]:
def ask_question(context, question):
    prompt = f"""
    Using ONLY the context below, answer the question as clearly as possible.
    If the answer is not present in the context, say so honestly.
    Context:
    {context}
    Question: {question}
    """
    response = model.generate_content(prompt)
    return response.text


def generate_report(summary, quiz, filename='study_report.txt'):
    lines = [
        'STUDY ASSISTANT REPORT',
        '=' * 50,
        '',
        'SUMMARY:',
        summary,
        ''
    ]

    lines.append('QUIZ:')

    if isinstance(quiz, list):
        for i, q in enumerate(quiz, 1):
            lines.append(f"Q{i}. {q['question']}")
            for k, v in q['options'].items():
                lines.append(f' {k}) {v}')
            lines.append(f" Correct Answer: {q['answer']}")
    else:
        lines.append(str(quiz))

    report = '\n'.join(lines)

    with open(filename, 'w') as f:
        f.write(report)

    print(f'Report saved as {filename}')
    return report


generate_report(summary, quiz)

print('\nYou can now ask questions about your notes.')
print("Type 'exit' to stop.\n")

while True:
    user_q = input('Your question: ')

    if user_q.strip().lower() == 'exit':
        print('Session ended. Goodbye!')
        break

    answer = ask_question(notes_text, user_q)
    print(f'\nAnswer: {answer}\n')


from google.colab import files
files.download('study_report.txt')

Report saved as study_report.txt

You can now ask questions about your notes.
Type 'exit' to stop.

Your question: What is Python?

Answer: Based on the context provided, Python is a programming language.

